In [4]:
import random
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, roc_auc_score
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_COLS = ['log_ret', 'rv_20', 'vol_z', 'mom_5']
QUANTILES = [0.75, 0.90]  



def prepare_data(ticker: str, start: str = '2016-01-01', seq_len: int = 30, horizon: int = 5,
                  split_frac: float = 0.8, val_frac: float = 0.15):
    """
    Chronological three-way split: train -> val -> test.
    Target: forward H-day realized volatility (see module docstring).
    """
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
    df['rv_20'] = df['log_ret'].rolling(20).std()
    df['vol_z'] = ((df['Volume'] - df['Volume'].rolling(20).mean())
                   / df['Volume'].rolling(20).std())
    df['mom_5'] = df['Close'].pct_change(5)

    sq_ret = df['log_ret'].values ** 2
    n = len(sq_ret)

    forward_vol = np.full(n, np.nan)
    for t in range(n - horizon):
        forward_vol[t] = np.sqrt(sq_ret[t + 1: t + 1 + horizon].sum())

    trailing_vol_H = np.sqrt(pd.Series(sq_ret).rolling(horizon).sum()).values

    df['forward_vol'] = forward_vol
    df['trailing_vol_H'] = trailing_vol_H
    df = df.dropna()

    df['target_ratio'] = df['forward_vol'] / df['trailing_vol_H']
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    raw_features = df[FEATURE_COLS].values
    raw_target = df['target_ratio'].values
    raw_persistence = df['trailing_vol_H'].values

    n = len(raw_features)
    train_end = int(split_frac * n)
    val_end = int((split_frac + (1 - split_frac) * val_frac) * n)

    feature_scaler = StandardScaler().fit(raw_features[:train_end])
    target_scaler = StandardScaler().fit(raw_target[:train_end].reshape(-1, 1))

    scaled_features = feature_scaler.transform(raw_features)
    scaled_target = target_scaler.transform(raw_target.reshape(-1, 1)).flatten()

    def make_windows(feat_arr, tgt_arr, seq):
        X = np.stack([feat_arr[i:i + seq] for i in range(len(feat_arr) - seq)])
        y = np.array([tgt_arr[i + seq] for i in range(len(feat_arr) - seq)])
        return X, y

    Xtr_np, ytr_np = make_windows(scaled_features[:train_end], scaled_target[:train_end], seq_len)
    Xval_np, yval_np = make_windows(scaled_features[train_end:val_end], scaled_target[train_end:val_end], seq_len)
    Xte_np, yte_np = make_windows(scaled_features[val_end:], scaled_target[val_end:], seq_len)

    to_tensor = lambda a: torch.tensor(a, dtype=torch.float32).to(device)
    Xtr, ytr = to_tensor(Xtr_np), to_tensor(ytr_np).unsqueeze(-1)
    Xval, yval = to_tensor(Xval_np), to_tensor(yval_np).unsqueeze(-1)
    Xte, yte = to_tensor(Xte_np), to_tensor(yte_np).unsqueeze(-1)

    baseline_tr  = np.ones(len(Xtr_np))
    baseline_val = np.ones(len(Xval_np))
    baseline_te  = np.ones(len(Xte_np))

    return {
        'Xtr': Xtr, 'ytr': ytr, 'Xval': Xval, 'yval': yval, 'Xte': Xte, 'yte': yte,
        'Xtr_np': Xtr_np, 'Xte_np': Xte_np,
        'baseline_tr': baseline_tr, 'baseline_val': baseline_val, 'baseline_te': baseline_te,
        'feature_scaler': feature_scaler, 'target_scaler': target_scaler,
        'df': df, 'seq_len': seq_len, 'horizon': horizon,
    }

def persistence_for_split(offset: int, n_windows: int) -> np.ndarray:
    """Persistence baseline for the RATIO target is 1.0 everywhere:
    'next H days' vol = last H days' vol' → ratio = 1."""
    return np.ones(n_windows)    


class PredictionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, seq_len):
        super().__init__()
        self.hidden_dim, self.num_layers = hidden_dim, num_layers
        self.seq_len = seq_len
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.lstm_path = nn.Linear(hidden_dim, output_dim)     
        self.linear_path = nn.Linear(input_dim * seq_len, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        lstm_out, _ = self.lstm(x, (h0, c0))
        nonlinear = self.lstm_path(lstm_out[:, -1, :])   
        flat_x = x.reshape(x.size(0), -1)
        linear = self.linear_path(flat_x)       
        return nonlinear + linear
       


def train_model(Xtr, ytr, Xval=None, yval=None, epochs=200, hidden_dim=24, num_layers=1,
                 lr=1e-3, print_every=25, patience=20, seq_len=30, linear_model=None):
    input_dim = Xtr.shape[-1]
    model = PredictionModel(input_dim, hidden_dim, num_layers, 1, seq_len).to(device)

    if linear_model is not None:
        with torch.no_grad():
            model.linear_path.weight.copy_(
                torch.tensor(linear_model.coef_, dtype=torch.float32).reshape(1, -1).to(device)
            )
            model.linear_path.bias.copy_(
                torch.tensor([linear_model.intercept_], dtype=torch.float32).to(device)
            )
    criterion = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        pred = model(Xtr)
        loss = criterion(pred, ytr)
        opt.zero_grad()
        loss.backward()
        opt.step()

        val_loss = None
        if Xval is not None:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(Xval), yval).item()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

        if epoch % print_every == 0:
            msg = f"  epoch {epoch:4d}  train_loss {loss.item():.6f}"
            if val_loss is not None:
                msg += f"  val_loss {val_loss:.6f}  (best {best_val_loss:.6f}, no_improve {epochs_no_improve})"
            print(msg)

        if Xval is not None and epochs_no_improve >= patience:
            if epoch % print_every != 0:
                print(f"  Early stopping at epoch {epoch} (no val improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def train_linear_baseline(Xtr_np: np.ndarray, ytr_np: np.ndarray) -> LinearRegression:
    n_samples = Xtr_np.shape[0]
    lr = LinearRegression()
    lr.fit(Xtr_np.reshape(n_samples, -1), ytr_np)
    return lr


def pct_vs_baseline(model_val: float, baseline_val: float, lower_is_better: bool):
    if baseline_val == 0:
        return None, "n/a (baseline is 0)"
    if lower_is_better:
        pct = (baseline_val - model_val) / abs(baseline_val) * 100
    else:
        pct = (model_val - baseline_val) / abs(baseline_val) * 100
    return pct, f"{pct:+.1f}%"


METRIC_DIRECTION = {'rmse': True}


def regime_discrimination(pred: np.ndarray, true: np.ndarray, quantiles=QUANTILES) -> dict:
    """AUC at each quantile threshold (label = "is this a top-Q realized-vol
    day?"), plus Spearman (threshold-free, computed once)."""
    result = {'by_quantile': {}}
    if np.std(pred) < 1e-8:
        result['spearman'] = float('nan')
    else:
        rho, _ = spearmanr(pred, true)
        result['spearman'] = rho

    for q in quantiles:
        threshold = np.quantile(true, q)
        label = (true > threshold).astype(int)
        if label.sum() == 0 or label.sum() == len(label):
            result['by_quantile'][q] = float('nan')
        else:
            result['by_quantile'][q] = roc_auc_score(label, pred)

    return result


def evaluate(model, Xte, yte, target_scaler, baseline_te, linear_model=None, Xte_np=None) -> dict:
    model.eval()
    with torch.no_grad():
        pred_scaled = model(Xte).cpu().numpy()
    true_scaled = yte.cpu().numpy()

    pred = target_scaler.inverse_transform(pred_scaled).flatten()
    true = target_scaler.inverse_transform(true_scaled).flatten()

    linear_pred = None
    if linear_model is not None and Xte_np is not None:
        n_samples = Xte_np.shape[0]
        linear_pred_scaled = linear_model.predict(Xte_np.reshape(n_samples, -1)).reshape(-1, 1)
        linear_pred = target_scaler.inverse_transform(linear_pred_scaled).flatten()

    results = {'_pred': pred, '_true': true, '_linear_pred': linear_pred, '_baseline_pred': baseline_te}

    def score(pred_arr, true_arr):
        return {
            'rmse': root_mean_squared_error(true_arr, pred_arr),
        }

    model_scores = score(pred, true)
    persistence_scores = score(baseline_te, true)
    linear_scores = score(linear_pred, true) if linear_pred is not None else None

    for m in ['rmse']:
        lower_better = METRIC_DIRECTION[m]
        pct_p, label_p = pct_vs_baseline(model_scores[m], persistence_scores[m], lower_better)
        entry = {
            'model': model_scores[m], 'baseline_persistence': persistence_scores[m],
            'pct_vs_persistence': pct_p, 'pct_vs_persistence_label': label_p,
            'beats_persistence': (model_scores[m] < persistence_scores[m]) if lower_better else (model_scores[m] > persistence_scores[m]),
        }
        if linear_scores is not None:
            pct_l, label_l = pct_vs_baseline(model_scores[m], linear_scores[m], lower_better)
            entry.update({
                'baseline_linear': linear_scores[m], 'pct_vs_linear': pct_l, 'pct_vs_linear_label': label_l,
                'beats_linear': (model_scores[m] < linear_scores[m]) if lower_better else (model_scores[m] > linear_scores[m]),
            })
        results[m] = entry

    results['regime_discrimination'] = {
        'model': regime_discrimination(pred, true),
        'persistence': regime_discrimination(baseline_te, true),
        'linear': regime_discrimination(linear_pred, true) if linear_pred is not None else None,
    }

    results['prediction_variance_ratio'] = pred.var() / true.var() if true.var() > 0 else float('nan')

    return results


def print_report(all_results):
    """Aggregate report across multiple seeds."""
    n = len(all_results)

    mean_model_rmse = np.mean([r['rmse']['model'] for r in all_results])
    mean_persistence_rmse = np.mean([r['rmse']['baseline_persistence'] for r in all_results])
    mean_linear_rmse = np.mean([r['rmse']['baseline_linear'] for r in all_results])

    mean_model_auc75 = np.mean([r['regime_discrimination']['model']['by_quantile'][0.75] for r in all_results])
    mean_model_auc90 = np.mean([r['regime_discrimination']['model']['by_quantile'][0.90] for r in all_results])
    mean_linear_auc75 = np.mean([r['regime_discrimination']['linear']['by_quantile'][0.75] for r in all_results])
    mean_linear_auc90 = np.mean([r['regime_discrimination']['linear']['by_quantile'][0.90] for r in all_results])

    mean_variance_ratio = np.mean([r['prediction_variance_ratio'] for r in all_results])

    print(f"\n{'='*60}")
    print(f"AGGREGATE REPORT ({n} seeds)")
    print('=' * 60)
    print(f"Average RMSE:")
    print(f"  Model:       {mean_model_rmse:.5f}")
    print(f"  Persistence: {mean_persistence_rmse:.5f}")
    print(f"  Linear:      {mean_linear_rmse:.5f}")
    print(f"\nAverage AUC:")
    print(f"  Model  @0.75: {mean_model_auc75:.3f}  @0.90: {mean_model_auc90:.3f}")
    print(f"  Linear @0.75: {mean_linear_auc75:.3f}  @0.90: {mean_linear_auc90:.3f}")
    print(f"\nAverage prediction variance ratio: {mean_variance_ratio:.3f}")
    print('=' * 60)
def block_bootstrap_rmse_improvement(pred: np.ndarray, baseline_pred: np.ndarray, true: np.ndarray,
                                      block_size: int = 30, n_boot: int = 1000) -> dict:
    """PRIMARY. 95% CI on (persistence_RMSE - model_RMSE) via block
    resampling (contiguous chunks, not single days -- respects the
    autocorrelation from overlapping windows)."""
    n = len(true)
    n_blocks = max(1, n // block_size)
    diffs = []
    for _ in range(n_boot):
        starts = np.random.randint(0, max(1, n - block_size), size=n_blocks)
        idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])
        model_rmse = root_mean_squared_error(true[idx], pred[idx])
        baseline_rmse = root_mean_squared_error(true[idx], baseline_pred[idx])
        diffs.append(baseline_rmse - model_rmse)

    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return {
        'observed_improvement': root_mean_squared_error(true, baseline_pred) - root_mean_squared_error(true, pred),
        'ci_95': (ci_low, ci_high),
        'significant': ci_low > 0,
    }


def block_bootstrap_auc_vs_persistence(pred: np.ndarray, baseline_pred: np.ndarray, true: np.ndarray,
                                        quantile: float = 0.75, block_size: int = 30, n_boot: int = 1000) -> dict:
    """
    SECONDARY. 95% CI on (model_AUC - persistence_AUC) at a given quantile
    -- beating chance-level 0.5 isn't the same as beating a baseline that
    already discriminates regimes reasonably well. This is the comparison
    that actually justifies the LSTM's added complexity, if it passes.
    """
    n = len(true)
    n_blocks = max(1, n // block_size)
    diffs = []
    for _ in range(n_boot):
        starts = np.random.randint(0, max(1, n - block_size), size=n_blocks)
        idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])
        threshold = np.quantile(true[idx], quantile)
        label = (true[idx] > threshold).astype(int)
        if label.sum() == 0 or label.sum() == len(label):
            continue
        model_auc = roc_auc_score(label, pred[idx])
        baseline_auc = roc_auc_score(label, baseline_pred[idx])
        diffs.append(model_auc - baseline_auc)

    if len(diffs) < 10:
        return {'ci_95': (float('nan'), float('nan')), 'significant': False,
                'note': 'Too few valid bootstrap samples (degenerate label splits).'}

    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

    full_label = (true > np.quantile(true, quantile)).astype(int)
    observed = roc_auc_score(full_label, pred) - roc_auc_score(full_label, baseline_pred)
    return {'observed_auc_diff': observed, 'ci_95': (ci_low, ci_high), 'significant': ci_low > 0}


def print_significance_report(rmse_boot: dict, auc_boot_by_q: dict):
    print(f"\n{'='*70}")
    print("SIGNIFICANCE TESTS (block bootstrap, ensemble)")
    print('=' * 70)

    print("\nPRIMARY -- RMSE improvement, model vs. persistence baseline:")
    print(f"  Observed improvement: {rmse_boot['observed_improvement']:.5f}")
    print(f"  95% CI: [{rmse_boot['ci_95'][0]:.5f}, {rmse_boot['ci_95'][1]:.5f}]  "
          f"({'SIGNIFICANT (CI excludes 0)' if rmse_boot['significant'] else 'not significant (CI includes 0)'})")

    print("\nSECONDARY -- AUC improvement, model vs. persistence baseline (not just vs. chance):")
    for q, auc_boot in auc_boot_by_q.items():
        print(f"  [quantile {q}]")
        if auc_boot.get('note'):
            print(f"    {auc_boot['note']}")
        else:
            print(f"    Observed AUC diff: {auc_boot['observed_auc_diff']:+.3f}")
            print(f"    95% CI: [{auc_boot['ci_95'][0]:+.3f}, {auc_boot['ci_95'][1]:+.3f}]  "
                  f"({'SIGNIFICANT' if auc_boot['significant'] else 'not significant'})")


def multi_seed_evaluation(data: dict, linear_model, n_seeds: int = 20) -> dict:
    all_preds = []
    all_results = []
    all_rmse_improvement = []
    all_auc = {q: [] for q in QUANTILES}
    last_results = None

    for seed in range(n_seeds):
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)

        model = train_model(
            data['Xtr'], data['ytr'],
            Xval=data['Xval'], yval=data['yval'],
            epochs=500, patience=20, print_every=10_000,
            seq_len=data['seq_len'], linear_model=linear_model,
        )
        results = evaluate(
            model, data['Xte'], data['yte'],
            data['target_scaler'], data['baseline_te'],
            linear_model=linear_model, Xte_np=data['Xte_np'],
        )
        last_results = results
        all_preds.append(results['_pred'])
        all_results.append(results)

        rmse_improve = results['rmse']['baseline_persistence'] - results['rmse']['model']
        all_rmse_improvement.append(rmse_improve)

        line = f"  seed {seed}: rmse_improvement={rmse_improve:+.5f}"
        for q in QUANTILES:
            auc = results['regime_discrimination']['model']['by_quantile'][q]
            all_auc[q].append(auc)
            line += f"  auc@{q}={auc:.3f}"
        print(line)

    all_rmse_improvement = np.array(all_rmse_improvement)

    print(f"\n{'='*70}")
    print(f"MULTI-SEED SUMMARY ({n_seeds} runs)")
    print('=' * 70)
    print(f"RMSE improvement over persistence: mean={all_rmse_improvement.mean():+.5f}, "
          f"std={all_rmse_improvement.std():.5f}, "
          f"positive in {(all_rmse_improvement > 0).sum()}/{n_seeds} runs")
    for q in QUANTILES:
        arr = np.array(all_auc[q])
        print(f"AUC@{q}: mean={arr.mean():.3f}, std={arr.std():.3f}, "
              f"above 0.5 in {(arr > 0.5).sum()}/{n_seeds} runs")

    rmse_ok = (all_rmse_improvement > 0).sum()
    auc_ok = {q: (np.array(all_auc[q]) > 0.5).sum() for q in QUANTILES}
    if rmse_ok <= 2 and all(v <= 2 for v in auc_ok.values()):
        print("\nImprovement/discrimination shows up in only a couple of seeds -- likely initialization luck, not real skill.")
    elif rmse_ok >= n_seeds * 0.7 and any(v >= n_seeds * 0.7 for v in auc_ok.values()):
        print("\nConsistent across most seeds on at least one metric -- more credible evidence of real skill.")
    else:
        print("\nMixed across seeds -- no reliable edge established yet.")

    print_report(all_results)
    # --- Ensemble significance test ---
    ensemble_pred = np.mean(all_preds, axis=0)
    true = last_results['_true']
    baseline_pred = last_results['_baseline_pred']

    print(f"\nEnsemble prediction stats: mean={ensemble_pred.mean():.4f}, "
          f"std={ensemble_pred.std():.4f}")

    rmse_boot = block_bootstrap_rmse_improvement(ensemble_pred, baseline_pred, true)
    auc_boot_by_q = {
        q: block_bootstrap_auc_vs_persistence(ensemble_pred, baseline_pred, true, quantile=q)
        for q in QUANTILES
    }
    print_significance_report(rmse_boot, auc_boot_by_q)

    return {'rmse_improvement': all_rmse_improvement, 'auc': all_auc,
            'ensemble_pred': ensemble_pred}


if __name__ == "__main__":
    TICKER = "NTES"
    SEQ_LEN = 30
    HORIZON = 5  

    print(f"Preparing data for {TICKER} (horizon={HORIZON} days)")
    data = prepare_data(TICKER, seq_len=SEQ_LEN, horizon=HORIZON)
    print(f"Train windows: {len(data['Xtr'])}, Val windows: {len(data['Xval'])}, Test windows: {len(data['Xte'])}")

    print("Training linear-regression baseline (same inputs)")
    linear_model = train_linear_baseline(data['Xtr_np'], data['ytr'].cpu().numpy().flatten())

    print("\n\n--- Multi-seed evaluation ---")
    multi_seed_evaluation(data, linear_model, n_seeds=20)

Preparing data for NTES (horizon=5 days)
Train windows: 2106, Val windows: 50, Test windows: 425
Training linear-regression baseline (same inputs)


--- Multi-seed evaluation ---
  epoch    0  train_loss 0.829472  val_loss 1.415743  (best 1.415743, no_improve 0)
  Early stopping at epoch 29 (no val improvement for 20 epochs)
  seed 0: rmse_improvement=+0.07153  auc@0.75=0.737  auc@0.9=0.755
  epoch    0  train_loss 0.826236  val_loss 1.432510  (best 1.432510, no_improve 0)
  Early stopping at epoch 24 (no val improvement for 20 epochs)
  seed 1: rmse_improvement=+0.06586  auc@0.75=0.733  auc@0.9=0.749
  epoch    0  train_loss 0.831125  val_loss 1.454863  (best 1.454863, no_improve 0)
  Early stopping at epoch 42 (no val improvement for 20 epochs)
  seed 2: rmse_improvement=+0.06516  auc@0.75=0.732  auc@0.9=0.746
  epoch    0  train_loss 0.827653  val_loss 1.438638  (best 1.438638, no_improve 0)
  Early stopping at epoch 30 (no val improvement for 20 epochs)
  seed 3: rmse_improvement=+